# Fake transportation trips

Anton Antonov   
July 2026

---

## Introduction

This notebook shows how to:

- Connect to the Model Management System (MMS) database of the Geo-Spatial Pricing Engine (GSPE)
- Take _raw_ transportation trips data for a given identifier
    - From the MMS table `raw_transportation_trips`
- Associate prices to each trip according to a formula (with complicated rules)
- Fill in the MMS table `transportation_trips`

---

## Setup

Packages for data frame manipulation and import:

In [2]:
import pandas as pd
import json

Using the (newer, recommended) modern `psycopg` (`psycopg3`):

In [3]:
import psycopg

In [4]:
DB_CONFIG = {
    'dbname': 'geo_spatial_pricing_engine',   # Change this
    'user': 'postgres',               # or your username
    'password': '',
    'host': 'localhost',              # or IP / remote host
    'port': '5432'
}

Connect to the database with:

```python
with psycopg.connect(**DB_CONFIG) as conn:
    ...
```

or similar.

Load the chatbook extension:

In [5]:
%load_ext JupyterChatbook

---

## Table information

Query to get all tables (excluding system tables):

In [6]:
with psycopg.connect(**DB_CONFIG) as conn:
    with conn.cursor() as cur:
        cur.execute("""
            SELECT table_schema, table_name, table_type
            FROM information_schema.tables
            WHERE table_schema NOT IN ('information_schema', 'pg_catalog')
            ORDER BY table_schema, table_name;
        """)
        
        tables = cur.fetchall()
        
        print(f"\n📋 Tables in '{DB_CONFIG['dbname']}':\n")
        for schema, table, ttype in tables:
            print(f"{schema:.<15} {table}")


📋 Tables in 'geo_spatial_pricing_engine':

public......... calibrated_value
public......... experiment
public......... experimental_result
public......... geo_taxonomy
public......... model
public......... model_parameter
public......... raw_transportation_trips
public......... tile_data
public......... transportation_trips


----

## Raw transportation trips data

In [17]:
with psycopg.connect(**DB_CONFIG) as conn:
    query = """
        SELECT *
        FROM raw_transportation_trips
        WHERE raw_transportation_trips_id = 'FAFDerived'
    """
    dfRawTrips = pd.read_sql(query, conn)

pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.


In [18]:
dfRawTrips

,id,raw_transportation_trips_id,start_lat,start_lon,start_state,start_city,start_zip_code,end_lat,end_lon,end_state,end_city,end_zip_code,distance,price
0,5,FAFDerived,42.665745,-73.798353,NewYork,Albany,None,33.762909,-84.422674,Georgia,Atlanta,None,1004.7467,0.0
1,6,FAFDerived,42.665745,-73.798353,NewYork,Albany,None,30.300536,-97.755044,Texas,Austin,None,1835.9657,0.0
2,7,FAFDerived,42.665745,-73.798353,NewYork,Albany,None,39.300213,-76.610516,Maryland,Baltimore,None,325.9673,0.0
3,8,FAFDerived,42.665745,-73.798353,NewYork,Albany,None,30.448453,-91.125899,Louisiana,BatonRouge,None,1481.5256,0.0
4,9,FAFDerived,42.665745,-73.798353,NewYork,Albany,None,30.084343,-94.145774,Texas,Beaumont,None,1667.8959,0.0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
4551,4556,FAFDerived,37.689363,-97.343805,Kansas,Wichita,None,27.967534,-82.475404,Florida,Tampa,None,1362.1461,0.0
4552,4557,FAFDerived,37.689363,-97.343805,Kansas,Wichita,None,32.154289,-110.871062,Arizona,Tucson,None,1143.3852,0.0
4553,4558,FAFDerived,37.689363,-97.343805,Kansas,Wichita,None,36.127949,-95.902316,Oklahoma,Tulsa,None,183.7673,0.0
4554,4559,FAFDerived,37.689363,-97.343805,Kansas,Wichita,None,36.779322,-76.024020,Virginia,VirginiaBeach,None,1367.7920,0.0


----

## Fake data prices

In [ ]:
%%chat_meta -i=python
clear

Cleared 2 messages of chat object python.

In [21]:
%%chat -i=python
Iterate over the rows of the data frame dfRawTrips, which has the columns:

id, start_lat, start_lon, end_lat, end_lon, distance

To each row apply the function:

```
priceFunc(start_lat, start_lon, end_lat, end_lon, distance, factor=0.2, offset=50)
```

Assign the result to the a new data frame dfFakeTrips.

```python
import pandas as pd

dfFakeTrips = dfRawTrips.copy()
dfFakeTrips["price"] = dfFakeTrips.apply(
    lambda row: priceFunc(
        row["start_lat"],
        row["start_lon"],
        row["end_lat"],
        row["end_lon"],
        row["distance"],
        factor=0.2,
        offset=50
    ),
    axis=1
)
```

Simple formula:

In [19]:
def priceFunc(start_lat, start_lon, end_lat, end_lon, distance, factor=0.2, offset=50):
    return distance * factor + offset

In [26]:
dfFakeTrips = dfRawTrips[["raw_transportation_trips_id", "start_lat", "start_lon", "end_lat", "end_lon", "distance"]].copy()
dfFakeTrips["price"] = dfFakeTrips.apply(
    lambda row: priceFunc(
        row["start_lat"],
        row["start_lon"],
        row["end_lat"],
        row["end_lon"],
        row["distance"],
        factor=0.2,
        offset=50
    ),
    axis=1
)

dfFakeTrips["transportation_trips_id"] = dfFakeTrips.apply(lambda row: "FAFDerivedLinear", axis=1)

dfFakeTrips

,raw_transportation_trips_id,start_lat,start_lon,end_lat,end_lon,distance,price,transportation_trips_id
0,FAFDerived,42.665745,-73.798353,33.762909,-84.422674,1004.7467,250.94934,FAFDerivedLinear
1,FAFDerived,42.665745,-73.798353,30.300536,-97.755044,1835.9657,417.19314,FAFDerivedLinear
2,FAFDerived,42.665745,-73.798353,39.300213,-76.610516,325.9673,115.19346,FAFDerivedLinear
3,FAFDerived,42.665745,-73.798353,30.448453,-91.125899,1481.5256,346.30512,FAFDerivedLinear
4,FAFDerived,42.665745,-73.798353,30.084343,-94.145774,1667.8959,383.57918,FAFDerivedLinear
...,...,...,...,...,...,...,...,...
4551,FAFDerived,37.689363,-97.343805,27.967534,-82.475404,1362.1461,322.42922,FAFDerivedLinear
4552,FAFDerived,37.689363,-97.343805,32.154289,-110.871062,1143.3852,278.67704,FAFDerivedLinear
4553,FAFDerived,37.689363,-97.343805,36.127949,-95.902316,183.7673,86.75346,FAFDerivedLinear
4554,FAFDerived,37.689363,-97.343805,36.779322,-76.024020,1367.7920,323.55840,FAFDerivedLinear


---

## Fill in DB table `transportation_trips`

In [ ]:
if False:
    with psycopg.connect(**DB_CONFIG) as conn:
        with conn.cursor() as cur:
            query = """
                INSERT INTO transportation_trips (
                    transportation_trips_id, 
                    raw_transportation_trips_id, 
                    start_lat, start_lon,
                    end_lat, end_lon, 
                    distance,
                    price,
                    is_training
                )
                VALUES (
                    %(transportation_trips_id)s, %(raw_transportation_trips_id)s,   
                    %(start_lat)s, %(start_lon)s,
                    %(end_lat)s, %(end_lon)s, 
                    %(distance)s,
                    %(price)s,
                    True
                )
            """
            cur.executemany(query, dfFakeTrips.to_dict("records"))
        conn.commit()